# Disaster Tweet Classification: Classical Machine Learning Benchmark

**Models:** Bag-of-Words (BoW) & TF-IDF (Word & Character n-grams) with Logistic Regression, Linear Support Vector Machine (LinearSVC), and Naive Bayes (MultinomialNB, ComplementNB)  
**Objective:** Comprehensive evaluation, per-model artifact organization, confusion matrix analysis, and multi-metric benchmarking across 10 humanitarian aid disaster categories.

---
### Notebook Structure
1. **Dataset Loading & Preprocessing Verification**
2. **Text Vectorization & Feature Engineering (BoW, Word TF-IDF, Char TF-IDF)**
3. **Model Training & Hyperparameter Setup (with Class Balancing)**
4. **Per-Model Artifact Serialization (`models/<model_name>/` and `saved_models/`)**
5. **Comprehensive Metrics Comparison (`metrics_comparison.csv` and `models_metrics_comparison.png`)**

In [ ]:
import os
import sys
import json
import string
import joblib
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

# Setup directories
DATA_DIR = Path("dataset")
RESULTS_DIR = Path("results/01_classical_ml_tfidf_bow")
MODELS_DIR = RESULTS_DIR / "saved_models"
PER_MODEL_DIR = RESULTS_DIR / "models"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PER_MODEL_DIR, exist_ok=True)

train_path = DATA_DIR / "train_clean.parquet"
val_path = DATA_DIR / "validation_clean.parquet"
test_path = DATA_DIR / "test_clean.parquet"

# Sourcing dataset from Kaggle or Google Drive
KAGGLE_INPUT_DIR = Path("/kaggle/input/humaid-disaster-tweets-parquet")
if KAGGLE_INPUT_DIR.exists():
    print("[+] Sourcing dataset from Kaggle dataset input...")
    for split in ["train", "validation", "test"]:
        p_clean = KAGGLE_INPUT_DIR / f"{split}_clean.parquet"
        p_raw = KAGGLE_INPUT_DIR / f"{split}.parquet"
        target_p = DATA_DIR / f"{split}_clean.parquet"
        if not target_p.exists():
            if p_clean.exists():
                pd.read_parquet(p_clean).to_parquet(target_p)
            elif p_raw.exists():
                pd.read_parquet(p_raw).to_parquet(target_p)

if not (train_path.exists() and val_path.exists() and test_path.exists()):
    raw_train = DATA_DIR / "train.parquet"
    raw_val = DATA_DIR / "validation.parquet"
    raw_test = DATA_DIR / "test.parquet"
    if not (raw_train.exists() and raw_val.exists() and raw_test.exists()):
        print("[+] Downloading HumAID dataset from Google Drive...")
        import gdown
        GDRIVE_URL = "https://drive.google.com/drive/folders/1pyMBc4SFc-sQvfmReiywPoN5cQbMpQBR?usp=drive_link"
        gdown.download_folder(url=GDRIVE_URL, output=str(DATA_DIR), quiet=False, use_cookies=False)

train_file = train_path if train_path.exists() else DATA_DIR / "train.parquet"
val_file = val_path if val_path.exists() else DATA_DIR / "validation.parquet"
test_file = test_path if test_path.exists() else DATA_DIR / "test.parquet"

train_df = pd.read_parquet(train_file)
val_df = pd.read_parquet(val_file)
test_df = pd.read_parquet(test_file)

text_col = "clean_text" if "clean_text" in train_df.columns else "tweet_text"
print(f"[+] Loaded splits using column '{text_col}':")
print(f"    Train: {len(train_df):,} samples")
print(f"    Validation: {len(val_df):,} samples")
print(f"    Test: {len(test_df):,} samples")

## 1. Data Preparation & Punctuation Normalization
For bag-of-words and TF-IDF linear classifiers, we strip trailing punctuation and lowercase tokens to reduce vocabulary sparsity while preserving semantic tokens.

In [ ]:
# Light punctuation stripping for classical linear models
def preprocess_for_classical(text):
    if not isinstance(text, str):
        return ""
    translator = str.maketrans("", "", string.punctuation)
    return text.translate(translator).lower()

X_train_raw = train_df[text_col].apply(preprocess_for_classical).values
X_val_raw = val_df[text_col].apply(preprocess_for_classical).values
X_test_raw = test_df[text_col].apply(preprocess_for_classical).values

# Encode labels
class_names = sorted(train_df["class_label"].unique())
label2idx = {name: i for i, name in enumerate(class_names)}
idx2label = {i: name for i, name in enumerate(class_names)}

y_train = train_df["class_label"].map(label2idx).values
y_val = val_df["class_label"].map(label2idx).values
y_test = test_df["class_label"].map(label2idx).values

print(f"[+] Encoded {len(class_names)} target classes:")
for i, name in enumerate(class_names):
    cnt = (y_train == i).sum()
    print(f"    Class {i:02d}: {name:<40} (Train Count: {cnt:,})")

## 2. Feature Extraction Pipelines
We construct 3 distinct feature representations:
1. **Bag-of-Words (CountVectorizer):** Unigrams + Bigrams with sublinear frequencies.
2. **Word TF-IDF Vectorizer:** Word n-grams (1, 2) with smooth inverse document frequency.
3. **Character TF-IDF Vectorizer:** Subword character n-grams (2, 5) to capture morphological patterns.

In [ ]:
print("[+] Fitting Feature Vectorizers...")

# 1. Bag of Words (CountVectorizer)
bow_vec = CountVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, max_features=30000)
X_train_bow = bow_vec.fit_transform(X_train_raw)
X_val_bow = bow_vec.transform(X_val_raw)
X_test_bow = bow_vec.transform(X_test_raw)
print(f"    BoW Feature Dim: {X_train_bow.shape[1]:,}")

# 2. Word TF-IDF Vectorizer
word_tfidf_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, sublinear_tf=True, max_features=30000)
X_train_word_tfidf = word_tfidf_vec.fit_transform(X_train_raw)
X_val_word_tfidf = word_tfidf_vec.transform(X_val_raw)
X_test_word_tfidf = word_tfidf_vec.transform(X_test_raw)
print(f"    Word TF-IDF Feature Dim: {X_train_word_tfidf.shape[1]:,}")

# 3. Char TF-IDF Vectorizer
char_tfidf_vec = TfidfVectorizer(analyzer='char', ngram_range=(2, 5), min_df=5, max_df=0.9, sublinear_tf=True, max_features=40000)
X_train_char_tfidf = char_tfidf_vec.fit_transform(X_train_raw)
X_val_char_tfidf = char_tfidf_vec.transform(X_val_raw)
X_test_char_tfidf = char_tfidf_vec.transform(X_test_raw)
print(f"    Char TF-IDF Feature Dim: {X_train_char_tfidf.shape[1]:,}")

# Save Vectorizers
joblib.dump(bow_vec, MODELS_DIR / "bow_vectorizer.joblib")
joblib.dump(word_tfidf_vec, MODELS_DIR / "word_tfidf_vectorizer.joblib")
joblib.dump(char_tfidf_vec, MODELS_DIR / "char_tfidf_vectorizer.joblib")
print("[+] Vectorizers saved to saved_models directory.")

## 3. Model Training, Per-Model Artifact Storage & Evaluation Suite
We train 10 classical models across the feature sets, handling class imbalance via balanced weighting and saving complete per-model evaluation folders.

In [ ]:
# Helper function to generate and save per-model evaluation plots and reports
def save_model_evaluation_artifacts(model_name, clf, y_true, y_pred, metrics_dict):
    model_slug = model_name.lower().replace(" ", "_").replace("+", "plus").replace("-", "_")
    curr_model_dir = PER_MODEL_DIR / model_slug
    os.makedirs(curr_model_dir, exist_ok=True)
    
    # 1. Classification Report
    report_str = classification_report(y_true, y_pred, target_names=class_names, digits=4)
    with open(curr_model_dir / "classification_report.txt", "w", encoding="utf-8") as f:
        f.write(f"=== Classification Report: {model_name} ===\n\n")
        f.write(report_str)
        
    # 2. Metrics JSON
    with open(curr_model_dir / "metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics_dict, f, indent=2)
        
    # 3. Dual Confusion Matrix Plot
    cm_raw = confusion_matrix(y_true, y_pred)
    cm_norm = cm_raw.astype('float') / cm_raw.sum(axis=1)[:, np.newaxis]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7), dpi=100)
    sns.heatmap(cm_raw, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax1)
    ax1.set_title(f"Raw Confusion Matrix: {model_name}", fontsize=11, fontweight='bold')
    ax1.set_xlabel("Predicted Label")
    ax1.set_ylabel("True Label")
    ax1.tick_params(axis='x', rotation=45)
    
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax2)
    ax2.set_title(f"Normalized Confusion Matrix: {model_name}", fontsize=11, fontweight='bold')
    ax2.set_xlabel("Predicted Label")
    ax2.set_ylabel("True Label")
    ax2.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig(curr_model_dir / "confusion_matrix.png", bbox_inches='tight')
    plt.close()
    
    # 4. Per-Class Precision, Recall, and F1 Bar Chart
    report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    per_class_df = pd.DataFrame([
        {
            "Class": cls,
            "Precision": report_dict[cls]["precision"],
            "Recall": report_dict[cls]["recall"],
            "F1-Score": report_dict[cls]["f1-score"],
            "Support": report_dict[cls]["support"]
        }
        for cls in class_names
    ])
    per_class_df.to_csv(curr_model_dir / "per_class_metrics.csv", index=False)
    
    fig, ax = plt.subplots(figsize=(12, 6), dpi=100)
    x = np.arange(len(class_names))
    width = 0.25
    ax.barh(x - width, per_class_df["Precision"], width, label="Precision", color="#3498db")
    ax.barh(x, per_class_df["Recall"], width, label="Recall", color="#2ecc71")
    ax.barh(x + width, per_class_df["F1-Score"], width, label="F1-Score", color="#e74c3c")
    ax.set_yticks(x)
    ax.set_yticklabels(class_names, fontsize=9)
    ax.set_xlabel("Score", fontsize=10, fontweight='bold')
    ax.set_title(f"Per-Class Performance: {model_name}", fontsize=12, fontweight='bold')
    ax.legend(loc='lower right')
    ax.set_xlim(0, 1.05)
    plt.tight_layout()
    plt.savefig(curr_model_dir / "per_class_metrics.png", bbox_inches='tight')
    plt.close()

# Model configurations
model_configs = [
    # BoW Models
    {"name": "BoW + MultinomialNB", "features": "bow", "model": MultinomialNB(alpha=0.1)},
    {"name": "BoW + ComplementNB", "features": "bow", "model": ComplementNB(alpha=0.1)},
    {"name": "BoW + LogisticRegression", "features": "bow", "model": LogisticRegression(class_weight='balanced', max_iter=1000, C=1.0, random_state=42)},
    {"name": "BoW + LinearSVC", "features": "bow", "model": LinearSVC(class_weight='balanced', max_iter=2000, C=1.0, random_state=42)},
    
    # Word TF-IDF Models
    {"name": "Word TF-IDF + MultinomialNB", "features": "word_tfidf", "model": MultinomialNB(alpha=0.1)},
    {"name": "Word TF-IDF + ComplementNB", "features": "word_tfidf", "model": ComplementNB(alpha=0.1)},
    {"name": "Word TF-IDF + LogisticRegression", "features": "word_tfidf", "model": LogisticRegression(class_weight='balanced', max_iter=1000, C=2.0, random_state=42)},
    {"name": "Word TF-IDF + LinearSVC", "features": "word_tfidf", "model": LinearSVC(class_weight='balanced', max_iter=2000, C=1.0, random_state=42)},
    
    # Char TF-IDF Models
    {"name": "Char TF-IDF + LogisticRegression", "features": "char_tfidf", "model": LogisticRegression(class_weight='balanced', max_iter=1000, C=2.0, random_state=42)},
    {"name": "Char TF-IDF + LinearSVC", "features": "char_tfidf", "model": LinearSVC(class_weight='balanced', max_iter=2000, C=1.0, random_state=42)},
]

feature_maps = {
    "bow": (X_train_bow, X_val_bow, X_test_bow),
    "word_tfidf": (X_train_word_tfidf, X_val_word_tfidf, X_test_word_tfidf),
    "char_tfidf": (X_train_char_tfidf, X_val_char_tfidf, X_test_char_tfidf),
}

eval_results = []

for cfg in model_configs:
    name = cfg["name"]
    feat_key = cfg["features"]
    clf = cfg["model"]
    
    X_tr, X_v, X_te = feature_maps[feat_key]
    print(f"\n[+] Training & Evaluating: {name}...")
    clf.fit(X_tr, y_train)
    
    # Predict
    y_val_pred = clf.predict(X_v)
    y_test_pred = clf.predict(X_te)
    
    # Evaluate
    val_acc = float(accuracy_score(y_val, y_val_pred))
    val_f1_macro = float(f1_score(y_val, y_val_pred, average='macro', zero_division=0))
    
    test_acc = float(accuracy_score(y_test, y_test_pred))
    test_f1_macro = float(f1_score(y_test, y_test_pred, average='macro', zero_division=0))
    test_f1_weighted = float(f1_score(y_test, y_test_pred, average='weighted', zero_division=0))
    test_prec_macro = float(precision_score(y_test, y_test_pred, average='macro', zero_division=0))
    test_rec_macro = float(recall_score(y_test, y_test_pred, average='macro', zero_division=0))
    
    metrics_dict = {
        "Model": name,
        "Feature Representation": feat_key.upper(),
        "Val Accuracy": val_acc,
        "Val Macro F1": val_f1_macro,
        "Test Accuracy": test_acc,
        "Test Macro F1": test_f1_macro,
        "Test Weighted F1": test_f1_weighted,
        "Test Macro Precision": test_prec_macro,
        "Test Macro Recall": test_rec_macro,
    }
    
    # Save Model Weights
    model_slug = name.lower().replace(" ", "_").replace("+", "plus").replace("-", "_")
    joblib.dump(clf, MODELS_DIR / f"{model_slug}.joblib")
    
    # Save Per-Model Artifacts (Reports & Plots)
    save_model_evaluation_artifacts(name, clf, y_test, y_test_pred, metrics_dict)
    
    eval_results.append(metrics_dict)
    print(f"    --> Test Macro F1: {test_f1_macro:.4f} | Test Accuracy: {test_acc:.4f}")

# Compile & Save Benchmark Comparison Table
results_df = pd.DataFrame(eval_results).sort_values(by="Test Macro F1", ascending=False)
results_df.to_csv(RESULTS_DIR / "metrics_comparison.csv", index=False)
print("\n=== Classical Machine Learning Models Comparison Table ===")
print(results_df.to_string(index=False))

# Plot Comparison Figure of All Models in this Notebook
fig, ax = plt.subplots(figsize=(14, 8), dpi=120)
sorted_df = results_df.sort_values(by="Test Macro F1", ascending=True)
y_pos = np.arange(len(sorted_df))

bars = ax.barh(y_pos, sorted_df["Test Macro F1"], color="#2980b9", edgecolor="black", height=0.6, label="Test Macro F1")
ax.plot(sorted_df["Test Accuracy"], y_pos, "ro", markersize=8, label="Test Accuracy")

for i, bar in enumerate(bars):
    f1_val = sorted_df.iloc[i]["Test Macro F1"]
    acc_val = sorted_df.iloc[i]["Test Accuracy"]
    ax.text(f1_val + 0.005, bar.get_y() + bar.get_height()/2, f"F1: {f1_val:.4f} | Acc: {acc_val:.4f}", va='center', fontsize=9, fontweight='bold')

ax.set_yticks(y_pos)
ax.set_yticklabels(sorted_df["Model"], fontsize=10, fontweight='bold')
ax.set_xlabel("Score", fontsize=11, fontweight='bold')
ax.set_title("Classical Machine Learning Benchmark: Test Macro F1 & Accuracy Comparison", fontsize=13, fontweight='bold', pad=15)
ax.set_xlim(0, 1.0)
ax.legend(loc="lower right", frameon=True)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "models_metrics_comparison.png", bbox_inches='tight')
plt.show()

print(f"[+] All artifacts, models, reports, and comparison plots saved to: {RESULTS_DIR.resolve()}")